# Code 10.7.4: The PPO update

Several epochs of shuffled minibatch updates on the combined loss: the clipped policy objective (with advantages normalized per minibatch), the value loss, and the entropy bonus. It also records the approximate KL and the clip fraction.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 10.7.1
import numpy as np

import torch

import torch.nn as nn


In [ ]:
def ppo_update(model, opt, batch, clip_eps, epochs, n_minibatches,
               vf_coef=0.5, ent_coef=0.01, max_grad_norm=0.5):
    """Several epochs of minibatch updates on the clipped PPO loss."""
    N = batch["obs"].shape[0]
    mb_size = N // n_minibatches
    kls, clipfracs = [], []
    for _ in range(epochs):
        perm = torch.randperm(N)
        for start in range(0, N, mb_size):
            idx = perm[start:start + mb_size]
            d = model.dist(batch["obs"][idx])
            logp = d.log_prob(batch["act"][idx])
            log_ratio = logp - batch["logp"][idx]
            ratio = log_ratio.exp()
            adv = batch["adv"][idx]
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)       # normalize per minibatch
            unclipped = ratio * adv
            clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
            policy_loss = -torch.min(unclipped, clipped).mean()
            value_loss = ((model.value(batch["obs"][idx]) - batch["ret"][idx]) ** 2).mean()
            entropy = d.entropy().mean()
            loss = policy_loss + vf_coef * value_loss - ent_coef * entropy
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
            with torch.no_grad():                              # diagnostics
                kls.append(((ratio - 1) - log_ratio).mean().item())
                clipfracs.append(((ratio - 1).abs() > clip_eps).float().mean().item())
    return np.mean(kls), np.mean(clipfracs)
